In [ ]:
import json

with open('assets/models.json', 'r') as f:
    models = json.load(f)

len(models)

1578

In [8]:
with_description = [model for model in models if model.get('intendedUse')]
len(with_description)

177

In [9]:
# write the filtered models to a new file
with open('assets/models_with_description.json', 'w') as f:
    json.dump(with_description, f, indent=4)

Create a vector database

In [ ]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from chromadb import Client, PersistentClient
import logging
from math import ceil
from itertools import batched

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

embedding_function = DefaultEmbeddingFunction()

client = PersistentClient(path='assets/chromadb')
db = client.get_or_create_collection("model_descriptions", embedding_function=embedding_function)

class BouncerBro:
    """ kicks you out if he sees you twice. """
    seen = set()
    def check(self, identifier):
        if identifier in self.seen:
            logger.error(f"seen {identifier} before!")
            return False
        self.seen.add(identifier)
        return True

# split descriptions into chunks of 100 words
def split_description(description, chunk_size=50):
    words = description.split()
    for i in range(0, len(words), chunk_size):
        yield ' '.join(words[i:i + chunk_size])

unique_bro = BouncerBro()

ids = []
documents = []
embeddings = []
metadatas = []

for doc in with_description:
    description = doc.get('intendedUse', '')
    
    identifier = doc.get('modelId', 'unknown')
    unique_bro.check(identifier)

    for n, chunk in enumerate(split_description(description)):
        metadatas.append(dict(document_id=identifier, block_id=n))
        documents.append(chunk)
        ids.append(identifier + str(n))

BATCH_SIZE = 64
for n, batch in enumerate(batched(documents, BATCH_SIZE)):
    print(f'working on batch {n+1} / {ceil(len(documents)/BATCH_SIZE)}')
    embeddings.append(embedding_function(batch))

working on batch 1 / 15
working on batch 2 / 15
working on batch 3 / 15
working on batch 4 / 15
working on batch 5 / 15
working on batch 6 / 15
working on batch 7 / 15
working on batch 8 / 15
working on batch 9 / 15
working on batch 10 / 15
working on batch 11 / 15
working on batch 12 / 15
working on batch 13 / 15
working on batch 14 / 15
working on batch 15 / 15
working on batch 16 / 15


In [50]:
import numpy as np
db.add(ids=ids, embeddings=np.concat(embeddings), documents=documents, metadatas=metadatas)